# E-Commerce Customer Churn — EDA & SQL Analysis
**Dataset:** ecommerce_customer_churn_dataset.csv (50,000 customers)  
**Goal:** Understand what drives customer churn and answer 10 business questions using Python EDA and SQL queries.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='muted')

# Load dataset
df = pd.read_csv('data/ecommerce_customer_churn_dataset.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Data Understanding

In [ ]:
# Column types and null counts
df.info()

In [ ]:
# Basic statistics
df.describe().round(2)

In [ ]:
# Missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
print('Missing values:')
print(missing)
print(f'\nTotal missing: {df.isnull().sum().sum()}')

## 3. Data Cleaning

In [ ]:
# Fill numeric nulls with median, categorical with mode
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# Remove duplicates
before = len(df)
df = df.drop_duplicates()
print(f'Duplicates removed: {before - len(df)}')
print(f'Clean dataset shape: {df.shape}')

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Overall churn rate
churn_rate = df['Churned'].value_counts(normalize=True) * 100
print('Churn Distribution:')
print(churn_rate.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df['Churned'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Churn Count', fontsize=13)
axes[0].set_xticklabels(['Active', 'Churned'], rotation=0)
axes[0].set_ylabel('Customers')

axes[1].pie(df['Churned'].value_counts(), labels=['Active', 'Churned'],
            autopct='%1.1f%%', colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Churn Rate', fontsize=13)
plt.tight_layout()
plt.savefig('data/churn_distribution.png', dpi=120)
plt.show()

In [ ]:
# Age distribution by churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, group in df.groupby('Churned'):
    name = 'Churned' if label == 1 else 'Active'
    group['Age'].plot(kind='kde', ax=axes[0], label=name)
axes[0].set_title('Age Distribution by Churn')
axes[0].legend()

# Gender vs churn
gender_churn = df.groupby('Gender')['Churned'].mean() * 100
gender_churn.plot(kind='bar', ax=axes[1], color=['steelblue', 'salmon'], edgecolor='black')
axes[1].set_title('Churn Rate by Gender')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_xticklabels(gender_churn.index, rotation=0)
plt.tight_layout()
plt.savefig('data/age_gender_churn.png', dpi=120)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
numeric_cols = df.select_dtypes(include='number')
corr = numeric_cols.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('data/correlation_heatmap.png', dpi=120)
plt.show()

In [ ]:
# Key numeric features vs Churn
key_features = ['Login_Frequency', 'Cart_Abandonment_Rate', 'Customer_Service_Calls',
                'Average_Order_Value', 'Lifetime_Value', 'Days_Since_Last_Purchase']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(key_features):
    df.boxplot(column=col, by='Churned', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('')
    axes[i].set_xticklabels(['Active', 'Churned'])

plt.suptitle('Key Features vs Churn', y=1.01, fontsize=14)
plt.tight_layout()
plt.savefig('data/features_vs_churn.png', dpi=120)
plt.show()

## 5. Load Data into SQLite

In [ ]:
# Create SQLite database and load the dataframe
conn = sqlite3.connect('data/ecommerce.db')
df.to_sql('customers', conn, if_exists='replace', index=True, index_label='customer_id')
print('Data loaded into SQLite successfully!')
print(f'Table: customers | Rows: {len(df)}')

## 6. Business Questions — SQL Queries

### Q1: What is the overall churn rate?

In [ ]:
query = """
SELECT 
    ROUND(SUM(Churned) * 100.0 / COUNT(*), 2) AS churn_rate_pct,
    SUM(Churned) AS total_churned,
    COUNT(*) - SUM(Churned) AS total_active,
    COUNT(*) AS total_customers
FROM customers
"""
pd.read_sql(query, conn)

### Q2: How does churn rate vary by country?

In [ ]:
query = """
SELECT 
    Country,
    COUNT(*) AS total_customers,
    SUM(Churned) AS churned,
    ROUND(SUM(Churned) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY Country
ORDER BY churn_rate_pct DESC
"""
result = pd.read_sql(query, conn)
print(result.to_string(index=False))

# Visualization
result.plot(kind='bar', x='Country', y='churn_rate_pct', color='tomato', edgecolor='black')
plt.title('Churn Rate by Country')
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('data/churn_by_country.png', dpi=120)
plt.show()

### Q3: Do customers with higher cart abandonment rates churn more?

In [ ]:
query = """
SELECT 
    CASE 
        WHEN Cart_Abandonment_Rate < 25 THEN 'Low (<25%)'
        WHEN Cart_Abandonment_Rate BETWEEN 25 AND 50 THEN 'Medium (25-50%)'
        WHEN Cart_Abandonment_Rate BETWEEN 50 AND 75 THEN 'High (50-75%)'
        ELSE 'Very High (>75%)'
    END AS abandonment_group,
    COUNT(*) AS total_customers,
    ROUND(SUM(Churned) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY abandonment_group
ORDER BY churn_rate_pct DESC
"""
pd.read_sql(query, conn)

### Q4: Does membership seniority protect against churn?

In [ ]:
query = """
SELECT 
    CASE 
        WHEN Membership_Years < 1 THEN '< 1 year'
        WHEN Membership_Years BETWEEN 1 AND 3 THEN '1-3 years'
        WHEN Membership_Years BETWEEN 3 AND 5 THEN '3-5 years'
        ELSE '> 5 years'
    END AS membership_group,
    COUNT(*) AS total_customers,
    ROUND(SUM(Churned) * 100.0 / COUNT(*), 2) AS churn_rate_pct,
    ROUND(AVG(Lifetime_Value), 2) AS avg_lifetime_value
FROM customers
GROUP BY membership_group
ORDER BY churn_rate_pct DESC
"""
pd.read_sql(query, conn)

### Q5: What is the average order value of churned vs. active customers?

In [ ]:
query = """
SELECT 
    CASE WHEN Churned = 1 THEN 'Churned' ELSE 'Active' END AS status,
    ROUND(AVG(Average_Order_Value), 2) AS avg_order_value,
    ROUND(AVG(Total_Purchases), 2) AS avg_purchases,
    ROUND(AVG(Lifetime_Value), 2) AS avg_lifetime_value
FROM customers
GROUP BY Churned
"""
pd.read_sql(query, conn)

### Q6: Does customer service call frequency predict churn?

In [ ]:
query = """
SELECT 
    CAST(Customer_Service_Calls AS INT) AS calls,
    COUNT(*) AS total_customers,
    ROUND(SUM(Churned) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY calls
ORDER BY calls
"""
result = pd.read_sql(query, conn)
print(result.to_string(index=False))

result.plot(kind='line', x='calls', y='churn_rate_pct', marker='o', color='tomato')
plt.title('Churn Rate by Number of Customer Service Calls')
plt.ylabel('Churn Rate (%)')
plt.xlabel('Customer Service Calls')
plt.tight_layout()
plt.savefig('data/churn_by_service_calls.png', dpi=120)
plt.show()

### Q7: Do high-discount users churn more?

In [ ]:
query = """
SELECT 
    CASE 
        WHEN Discount_Usage_Rate < 25 THEN 'Low (<25%)'
        WHEN Discount_Usage_Rate BETWEEN 25 AND 50 THEN 'Medium (25-50%)'
        WHEN Discount_Usage_Rate BETWEEN 50 AND 75 THEN 'High (50-75%)'
        ELSE 'Very High (>75%)'
    END AS discount_group,
    COUNT(*) AS total_customers,
    ROUND(SUM(Churned) * 100.0 / COUNT(*), 2) AS churn_rate_pct,
    ROUND(AVG(Average_Order_Value), 2) AS avg_order_value
FROM customers
GROUP BY discount_group
ORDER BY churn_rate_pct DESC
"""
pd.read_sql(query, conn)

### Q8: How does login frequency impact churn?

In [ ]:
query = """
SELECT 
    CASE 
        WHEN Login_Frequency < 5 THEN 'Low (<5)'
        WHEN Login_Frequency BETWEEN 5 AND 15 THEN 'Medium (5-15)'
        WHEN Login_Frequency BETWEEN 15 AND 25 THEN 'High (15-25)'
        ELSE 'Very High (>25)'
    END AS login_group,
    COUNT(*) AS total_customers,
    ROUND(SUM(Churned) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY login_group
ORDER BY churn_rate_pct DESC
"""
pd.read_sql(query, conn)

### Q9: Which signup quarter has the highest long-term churn?

In [ ]:
query = """
SELECT 
    Signup_Quarter,
    COUNT(*) AS total_customers,
    SUM(Churned) AS churned,
    ROUND(SUM(Churned) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY Signup_Quarter
ORDER BY churn_rate_pct DESC
"""
pd.read_sql(query, conn)

### Q10: Who are the top 10 highest-value customers at risk of churning?

In [ ]:
query = """
SELECT 
    customer_id,
    Country,
    Age,
    Lifetime_Value,
    Total_Purchases,
    Customer_Service_Calls,
    Cart_Abandonment_Rate,
    Days_Since_Last_Purchase
FROM customers
WHERE Churned = 0
    AND Days_Since_Last_Purchase > 60
    AND Customer_Service_Calls >= 5
    AND Cart_Abandonment_Rate > 60
ORDER BY Lifetime_Value DESC
LIMIT 10
"""
pd.read_sql(query, conn)

## 7. Key Findings Summary

In [ ]:
# Summary statistics by churn status
summary = df.groupby('Churned')[[
    'Age', 'Login_Frequency', 'Cart_Abandonment_Rate',
    'Customer_Service_Calls', 'Lifetime_Value', 'Days_Since_Last_Purchase'
]].mean().round(2)
summary.index = ['Active', 'Churned']
print('Mean values — Active vs. Churned:')
summary

In [ ]:
conn.close()
print('Analysis complete. Database connection closed.')